# Rijksmuseum

### Exploration of Collection

There are two ways of comunication with the database:
1) OAI-PHM: https://data.rijksmuseum.nl/docs/oai-pmh/
2) JSON-LD: https://data.rijksmuseum.nl/docs/ldes/

I do not really know the difference in quality, neither if one is more straight forward than the other.

### Set Up

Create a session, with retries and "polite" headers, which restarts after ```ROTATE_EVERY=25``` requests. XML namespase bindings as a dictionary. As well create a cache for the requests.

In [279]:
import os, json, hashlib, itertools
import requests, xml.etree.ElementTree as ET
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE = "https://data.rijksmuseum.nl/oai"

NS = {
    "oai": "http://www.openarchives.org/OAI/2.0/",
    "rdf": "http://www.w3.org/1999/02/22-rdf-syntax-ns#",
    "dc":  "http://purl.org/dc/elements/1.1/",
    "dct": "http://purl.org/dc/terms/",
    "dcterms": 'http://purl.org/dc/terms/',
    "edm": "http://www.europeana.eu/schemas/edm/",
    "ore": "http://www.openarchives.org/ore/terms/",
    "skos":"http://www.w3.org/2004/02/skos/core#",
    "rdfs":"http://www.w3.org/2000/01/rdf-schema#",
}

def new_session():
    s = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=0.6,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    s.mount("https://", HTTPAdapter(max_retries=retry))
    s.headers.update({
        "User-Agent": "rijks-notebook/0.1 (ex@mple.com)",
        "Accept": "application/xml",
        "Connection": "close",
    })
    return s

SESSION = new_session()

CACHE_DIR = "cache/rijks-oai"
os.makedirs(CACHE_DIR, exist_ok=True)
_request_counter = itertools.count()
ROTATE_EVERY = 25

def _cache_path(params):
    key = json.dumps(params, sort_keys=True)
    h = hashlib.md5(key.encode("utf-8")).hexdigest()
    return os.path.join(CACHE_DIR, f"{h}.xml")

# Now cached
def fetch(params, timeout=30, session=SESSION, force=False):
    cache_file = _cache_path(params)
    if not force and os.path.exists(cache_file):
        with open(cache_file, "rb") as f:
            return ET.fromstring(f.read())

    i = next(_request_counter)
    if i > 0 and i % ROTATE_EVERY == 0:
        try:
            session.close()
        except Exception:
            pass
        session = new_session()

    r = session.get(BASE, params=params, timeout=timeout)
    print(f"GET {r.url} -> {r.status_code}")
    r.raise_for_status()
    content = r.content
    with open(cache_file, "wb") as f:
        f.write(content)
    return ET.fromstring(content)

##### Checking the first 20 datapoints
Using the OAI-PMH to see what information is available. Try to extract te image from it.

In [280]:
def extract_image_from_aggregation(rdf_root):
    aggr = rdf_root.find(".//ore:Aggregation", NS)
    if aggr is None:
        return None
    # Case A: resource attribute
    isb = aggr.find(".//edm:isShownBy", NS)
    if isb is not None:
        res = isb.attrib.get(f"{{{NS['rdf']}}}resource")
        if res:
            return res
    # Case B: nested WebResource/@rdf:about
    wr = aggr.find(".//edm:isShownBy/edm:WebResource", NS)
    if wr is not None:
        return wr.attrib.get(f"{{{NS['rdf']}}}about")
    return None

items, token = [], None
while len(items) < 20:
    params = {"verb": "ListRecords", "metadataPrefix": "edm"} if not token else {"verb": "ListRecords", "resumptionToken": token}
    root = fetch(params)

    for rec in root.findall(".//oai:record", NS):
        md = rec.find(".//oai:metadata", NS)
        if md is None:
            continue
        rdf = md.find(".//{http://www.w3.org/1999/02/22-rdf-syntax-ns#}RDF")
        if rdf is None:
            continue

        titles = [el.text for el in rdf.findall(".//dc:title", NS) if el.text]
        creators = [el.text for el in rdf.findall(".//dc:creator", NS) if el.text]
        types = [el.text for el in rdf.findall(".//dc:type", NS) if el.text]
        formats = [el.text for el in rdf.findall(".//dc:format", NS) if el.text]

        image_url = extract_image_from_aggregation(rdf)

        items.append({
            "title": (titles[0] if titles else None),
            "creator": (creators[0] if creators else None),
            "image": image_url,
            "type": types,
            "format": formats,
        })

        if len(items) >= 20:
            break

    # pagination, very important!
    rt = root.find(".//{http://www.openarchives.org/OAI/2.0/}resumptionToken")
    token = rt.text if rt is not None and rt.text else None
    if not token:
        break

for i, it in enumerate(items, 1):
    print(i, it)

1 {'title': 'Before, Behind, Between, Above, Below', 'creator': None, 'image': 'https://iiif.micr.io/KMFvF/full/max/0/default.jpg', 'type': [], 'format': []}
2 {'title': 'Badende jongens', 'creator': None, 'image': 'https://iiif.micr.io/wgIZp/full/max/0/default.jpg', 'type': [], 'format': []}
3 {'title': 'Portret van Dirk I, graaf van Holland', 'creator': None, 'image': 'https://iiif.micr.io/owHmZ/full/max/0/default.jpg', 'type': [], 'format': []}
4 {'title': 'Peso of acht reaal uit de republiek Mexico, 1863, geslagen te Real de Catorce, waarbij in China op de voorzijde vijf - en op de keerzijde zes sinogrammen zijn ingestempeld', 'creator': None, 'image': 'https://iiif.micr.io/NVgFC/full/max/0/default.jpg', 'type': [], 'format': []}
5 {'title': 'Wilhelmina Koningin der Nederlanden', 'creator': None, 'image': 'https://iiif.micr.io/PPxWZ/full/max/0/default.jpg', 'type': [], 'format': []}
6 {'title': 'De Kroonprins van Oranje in de Slag bij Waterloo 1815', 'creator': None, 'image': 'http

### Exploration of ListSets

All objects belong to either one or more categories or ListSets. They are important if we want to filter in/out any particular category. There are 192 different and overlapping ListSets. Some important ones are:

  - 260239  (Entire Public Domain Set)
  - 260245  (Aziatische keramiek / Asian ceramics)
  - 2611  (Cat. 15e en 16e eeuwse Nederlandse Schilderijen / Cat. 15th and 16th century Dutch Paintings)
  - 26113  ((under construction) European Sculpture)
  - 26118  (Flemish Paintings in the Rijksmuseum)
  - 2612  (Cat. 17e eeuwse Nederlandse Schilderijen I / Cat. 17th century Dutch Paintings I)
  - 261208  (schilderijen / paintings)
  - 26121  (Dutch Paintings of the Seventeenth Century in the Rijksmuseum)
  - 261223  (Aziatische keramiek / Asian ceramics)
  - 261224  (Europees keramiek / European ceramics)
  - 261231  (keramiek)
  - 261233  (porselein / porcelain)
  - 26126  (beeldhouwwerken / sculptures)
  - 2613  (Cat. 17e eeuwse Vlaamse Schilderijen / Cat. 17th century Flemish Paintings)
  - 26142  (Hollands porselein / Dutch porcelain)
  - 26147  (European porcelain / Europees porselein)
  - 2616  (Early Netherlandish Paintings)
  - 2618  (European Sculpture in the Rijksmuseum)
  - 26191  (Midden-Oosten keramiek)
  
Theres naturally way more, which might also be useful. The full list is the output of the following cell.

In [281]:
import re

BASE = "https://data.rijksmuseum.nl/oai"

IMPORTANT_PAT = re.compile(r"\b(public|domain|painting|paintings|schilderij|schilderijen|sculpture|beeldhouwwerken|keramiek|porselein|porcelain)\b", re.I)

def list_sets():
    token = None
    seen = 0
    candidates = []
    while True:
        params = {"verb": "ListSets"} if not token else {"verb": "ListSets", "resumptionToken": token}
        root = fetch(params)
        for s in root.findall(".//oai:set", NS):
            spec = (s.findtext("oai:setSpec", default="", namespaces=NS) or "").strip()
            name = (s.findtext("oai:setName", default="", namespaces=NS) or "").strip()
            seen += 1
            mark = " possible important set" if IMPORTANT_PAT.search(name) else ""
            print(f"{seen:5d}. setSpec={spec} | setName={name}{mark}")
            if mark:
                candidates.append((spec, name))
        rt = root.find(".//oai:resumptionToken", NS)
        token = rt.text.strip() if (rt is not None and rt.text) else None
        if not token:
            break
    return candidates


print("Listing OAI-PMH sets from Rijksmuseum…")
painting_sets = list_sets()
if painting_sets:
    print("\nInteresting sets:")
    for spec, name in painting_sets:
        print(f"  - {spec}  ({name})")

Listing OAI-PMH sets from Rijksmuseum…
    1. setSpec=260210 | setName=Against Opacity
    2. setSpec=260211 | setName=Indonesian heritage / Indisch erfgoed
    3. setSpec=260212 | setName=RijksXL
    4. setSpec=260213 | setName=Top 100
    5. setSpec=260214 | setName=Top 1000
    6. setSpec=260215 | setName=Vereniging Rembrandt
    7. setSpec=260216 | setName=Top 100 prentenkabinet / Top 100 print room
    8. setSpec=260235 | setName=sieraden / jewellery
    9. setSpec=260236 | setName=mode / fashion
   10. setSpec=260239 | setName=Entire Public Domain Set possible important set
   11. setSpec=260241 | setName=biografic / biografisch
   12. setSpec=260242 | setName=Dutch Delftware / Delfts aardewerk
   13. setSpec=260243 | setName=birdwatching / vogelen
   14. setSpec=260244 | setName=MIMO: Musical Instruments
   15. setSpec=260245 | setName=Aziatische keramiek / Asian ceramics possible important set
   16. setSpec=260246 | setName=Staten-Generaal
   17. setSpec=260247 | setName=frien

### How is an RDF node composed

It seems as if most of the info is already readily available from the RDF. For a reason I do not yet understand, the set parameter sometimes (coould be many times) causes a timeout. Maybe is just a loadbalancing problem. I think it would be wise to try to circumvent it, if possible. I think, one problem might be the size of the set. For example, 260239 (Entire Public Domain), which is probably huge, breaks. But 26191 (Midden-Oosten keramiek), does well.

In [284]:
# Using "set" might break the query.
root = fetch({"verb":"ListRecords", "metadataPrefix":"edm", "set":"26191"})

rec = root.find(".//oai:record", NS)
md = rec.find(".//oai:metadata", NS)
rdf = md.find(".//rdf:RDF", NS)

def walk(elem, path="rdf:RDF"):
    tag = elem.tag.split("}")[-1]  # localname
    here = f"{path}/{tag}"
    attribs = {k.split('}')[-1]: v for k,v in elem.attrib.items()}
    text = (elem.text or "").strip()
    if attribs or text:
        print(here, "| ATTR:", attribs, "| TEXT:", (text[:80]+"..." if len(text)>80 else text))
    for child in list(elem):
        walk(child, here)
for rdf in root.findall(".//oai:record", NS):
    md = rdf.find(".//oai:metadata", NS)

walk(rdf)

GET https://data.rijksmuseum.nl/oai?verb=ListRecords&metadataPrefix=edm&set=26191 -> 200
rdf:RDF/record/header/identifier | ATTR: {} | TEXT: https://id.rijksmuseum.nl/200415740
rdf:RDF/record/header/datestamp | ATTR: {} | TEXT: 2025-06-05T17:34:43Z
rdf:RDF/record/header/setSpec | ATTR: {} | TEXT: 260239
rdf:RDF/record/header/setSpec | ATTR: {} | TEXT: 261223
rdf:RDF/record/header/setSpec | ATTR: {} | TEXT: 26191
rdf:RDF/record/metadata/RDF/Aggregation | ATTR: {'about': 'https://id.rijksmuseum.nl/200415740#aggregation'} | TEXT: 
rdf:RDF/record/metadata/RDF/Aggregation/aggregatedCHO/ProvidedCHO | ATTR: {'about': 'https://id.rijksmuseum.nl/200415740'} | TEXT: 
rdf:RDF/record/metadata/RDF/Aggregation/aggregatedCHO/ProvidedCHO/creator | ATTR: {'resource': 'https://id.rijksmuseum.nl/210640'} | TEXT: 
rdf:RDF/record/metadata/RDF/Aggregation/aggregatedCHO/ProvidedCHO/description | ATTR: {'lang': 'en'} | TEXT: Medieval potters in Iran produced expensive wares with lustre decoration. Lustre...
r

### Get information from JSON-LD

Some information is not available form the RDF directly. In fact, quite some of the information sits behind the JSON-LD in the respective links. However, the image is only on the RDF, to my understanding. In the following cells I would try to get one full record with the most amount of information combining both RDF + JSON-LD. Here there is an example with an artistID, which is important for the following part.

In [320]:
import json

LD_HEADERS = {
    "Accept": "application/ld+json, application/json;q=0.9",
}

LD_CACHE_DIR = "cache/rijks-jsonld"
os.makedirs(LD_CACHE_DIR, exist_ok=True)

def _ld_cache_path(uri):
    h = hashlib.md5(uri.encode("utf-8")).hexdigest()
    return os.path.join(LD_CACHE_DIR, f"{h}.json")

def fetch_ld(uri, session = SESSION, timeout = 30, force = False):
    cache_file = _ld_cache_path(uri)

    if not force and os.path.exists(cache_file):
        try:
            with open(cache_file, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass 

    r = session.get(uri, headers=LD_HEADERS, timeout=timeout, allow_redirects=True)
    r.raise_for_status()

    ctype = (r.headers.get("Content-Type") or "").lower()
    if "json" in ctype:
        try:
            data = r.json()
        except ValueError:

            data = json.loads(r.content.decode(r.encoding or "utf-8", errors="ignore"))
        try:
            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False)
        except Exception:
            pass
        return data
    return None

person = fetch_ld("https://id.rijksmuseum.nl/2103209")
#print(json.dumps(person, indent=2, ensure_ascii=False))


### Extracting Artist Information
One important thing is that sometimes information is available either, only in Dutch, or in English. Its important to extract both, just in case. So the artist information covers:
- Names
- Date of Birth
- Place of Birth
- Date of Death
- Place of Death
- Activities
- Other links

In [321]:
from functools import lru_cache

AAT_LANG_EN = "http://vocab.getty.edu/aat/300388277"  # English
AAT_LANG_NL = "http://vocab.getty.edu/aat/300388256"  # Dutch

def lang_tags(node):
    tags = []
    for lang in (node.get("language") or []):
        lid = lang.get("id") or ""
        if lid == AAT_LANG_EN: tags.append("English")
        elif lid == AAT_LANG_NL: tags.append("Dutch")
        else: tags.append("und")
    return tags or ["und"]

def labels_by_lang(items):
    out = {}
    for it in (items or []):
        if not isinstance(it, dict): 
            continue
        if it.get("type") != "Name": 
            continue
        content = (it.get("content") or "").strip()
        if not content: 
            continue
        for tag in lang_tags(it):
            out.setdefault(tag, [])
            if content not in out[tag]:
                out[tag].append(content)
    return out

@lru_cache(maxsize=10000)
def read_rijksmuseum_place(uri):
    data = fetch_ld(uri)
    if not isinstance(data, dict):
        return None
    return labels_by_lang(data.get("identified_by", []))

@lru_cache(maxsize=10000)
def read_rijksmuseum_concept(uri):
    if not uri: 
        return None
    try:
        data = fetch_ld(uri)
    except requests.exceptions.RequestException as e:
        print(e)
        return None
    if not isinstance(data, dict): 
        return None
    return labels_by_lang(data.get("identified_by"))

@lru_cache(maxsize=10000)
def read_rijksmuseum_person(uri, depth = 1):
    data = fetch_ld(uri)
    if not isinstance(data, dict):
        return None

    result = {
        "id": uri,
        "names_by_lang": labels_by_lang(data.get("identified_by")),
        "birth": {"date": None, "place_by_lang": None},
        "death": {"date": None, "place_by_lang": None},
        "activities_by_lang": {}, 
        "equivalent": [e.get("id") for e in (data.get("equivalent") or []) if isinstance(e, dict) and e.get("id")] # Other links
    }

    # Birth info
    born = data.get("born")
    if isinstance(born, dict):
        ts = born.get("timespan") or {}
        result["birth"]["date"] = ts.get("begin_of_the_begin") or ts.get("end_of_the_end")
        if depth > 0:
            for pl in (born.get("took_place_at") or []):
                if isinstance(pl, dict) and pl.get("id"):
                    labels = read_rijksmuseum_place(pl["id"])
                    if labels:
                        result["birth"]["place_by_lang"] = labels
                        break

    # Death info
    died = data.get("died")
    if isinstance(died, dict):
        ts = died.get("timespan") or {}
        result["death"]["date"] = ts.get("end_of_the_end") or ts.get("begin_of_the_begin")
        if depth > 0:
            for pl in (died.get("took_place_at") or []):
                if isinstance(pl, dict) and pl.get("id"):
                    labels = read_rijksmuseum_place(pl["id"])
                    if labels:
                        result["death"]["place_by_lang"] = labels
                        break

    # Activities the artist/manufacturer performed (i.e painter, sculpturer, etc.)
    act_labels = {}
    for act in (data.get("carried_out") or []):
        for cl in (act.get("classified_as") or []):
            uri = cl.get("id")
            if not uri:
                continue
            if depth > 0:
                labs = read_rijksmuseum_concept(uri) or {}
                for lang, texts in labs.items():
                    act_labels.setdefault(lang, set()).update(texts)


    result["activities_by_lang"] = {k: sorted(v) for k, v in act_labels.items()}

    return result

# Example usage
person = read_rijksmuseum_person("https://id.rijksmuseum.nl/2103209")
#print(json.dumps(person, indent=2, ensure_ascii=False))

### Extract information for one record using the JSON-LD

The information needed (i.e description, inscription, subjects), is not always on the same part of the JSON. One has to consider different paths per attribute.

In [322]:
# Languages
AAT_LANG_EN = "http://vocab.getty.edu/aat/300388277"
AAT_LANG_NL = "http://vocab.getty.edu/aat/300388256"

# Text types (i.e description, dimensions, etc.)
DESC_AAT  = {"http://vocab.getty.edu/aat/300048722"}
DIM_AAT   = {"http://vocab.getty.edu/aat/300435430"}
MED_AAT   = {"http://vocab.getty.edu/aat/300435429"}
CRED_AAT  = {"http://vocab.getty.edu/aat/300026687"}
PROV_AAT  = {"http://vocab.getty.edu/aat/300444174"}
LABEL_AAT = {"http://vocab.getty.edu/aat/300417268"}
INSCRIPTION_AAT = {"http://vocab.getty.edu/aat/300435414"}
REJECTED_AAT    = {"http://vocab.getty.edu/aat/300404908"}

# Units
UNIT_LABELS = {
    "http://vocab.getty.edu/aat/300379098": "cm",
    "http://vocab.getty.edu/aat/300379100": "mm",
    "http://vocab.getty.edu/aat/300379097": "m",
}

# For extracting dimensions
FRAME_MARKERS = ("frame", "outer size", "buitenmaat", "depth", "diepte", "thickness", "dikte")

##### Some helper functions

In [323]:
def as_list(x):
    if x is None:
        return []
    return x if isinstance(x, list) else [x]

def dedupe_keep_order(items):
    seen, out = set(), []
    for it in items:
        if it not in seen:
            seen.add(it); out.append(it)
    return out

def merge_multilang_dicts(dicts):
    merged = {}
    for d in dicts or []:
        if not isinstance(d, dict): 
            continue
        for lang, vals in d.items():
            if not vals: 
                continue
            merged.setdefault(lang, [])
            for v in vals:
                if v not in merged[lang]:
                    merged[lang].append(v)
    return merged

def lang_tags(lang_list):
    tags = []
    for lang in (lang_list or []):
        lid = (lang.get("id") or "").strip()
        if lid == AAT_LANG_EN: tags.append("English")
        elif lid == AAT_LANG_NL: tags.append("Dutch")
        else: tags.append("und")
    return tags or ["und"]

def lang_code(node):
    lids = [ (l.get("id") or "") for l in (node.get("language") or []) if isinstance(l, dict) ]
    if AAT_LANG_EN in lids: return "English"
    if AAT_LANG_NL in lids: return "Dutch"
    return None

def classified_ids(node):
    out = set()
    for c in (node.get("classified_as") or []):
        if isinstance(c, dict) and c.get("id"): out.add(c["id"])
        elif isinstance(c, str): out.add(c)
    return out

# Traverses nested LinguisticObjects, because language sometimes appears only in the "parent" node
def walk_lo_with_lang(node, inherited=None):
    if not isinstance(node, dict):
        return
    here_lang = node.get("language") or inherited
    if node.get("type") == "LinguisticObject" and (node.get("content") or "").strip():
        yield node, here_lang
    for ch in (node.get("part") or []):
        yield from walk_lo_with_lang(ch, here_lang)

import html

# Some descriptions have HTML tags on it, has to be removed.
_BR_RE  = re.compile(r'(?is)<\s*br\s*/?\s*>')
_TAG_RE = re.compile(r'(?is)<[^>]+>')
def strip_html(text, collapse=True):
    if not isinstance(text, str): return ""
    t = _BR_RE.sub(" ", text)
    t = _TAG_RE.sub("", t)
    t = html.unescape(t).replace("\xa0", " ")
    if collapse:
        t = "\n".join(" ".join(line.split()) for line in t.splitlines()).strip()
    return t

In [325]:
# Titles
def labels_by_lang(name_nodes):
    out = {"English": [], "Dutch": []}
    for n in name_nodes or []:
        if not isinstance(n, dict) or n.get("type") != "Name": 
            continue
        txt = (n.get("content") or "").strip()
        if not txt: continue
        for tag in lang_tags(n.get("language")):
            if tag in ("English", "Dutch") and txt not in out[tag]:
                out[tag].append(txt)
    return {k:v for k,v in out.items() if v}

def pick_titles_by_lang(cho_ld):
    return labels_by_lang([n for n in (cho_ld.get("identified_by") or []) if n.get("type") == "Name"])

def pick_titles(cho_ld):
    return pick_titles_by_lang(cho_ld)

# Artist Info
def _aa_is_rejected(aa):
    return bool(classified_ids(aa) & REJECTED_AAT)

def pick_artist_info(cho_ld):
    def add_from_cob(cob, bag):
        for a in as_list(cob):
            if isinstance(a, dict) and a.get("id") and a.get("type") in ("Person", "Actor"):
                bag.append((a["id"], None))

    def add_from_assignment(aa, bag):
        if not isinstance(aa, dict) or aa.get("assigned_property") != "carried_out_by":
            return
        if _aa_is_rejected(aa): 
            return
        for a in as_list(aa.get("assigned")):
            if not isinstance(a, dict): 
                continue
            if a.get("id") and a.get("type") in ("Person", "Actor"):
                bag.append((a["id"], None))
            elif a.get("type") == "Production":
                add_from_cob(a.get("carried_out_by"), bag)
            elif a.get("type") == "Group":  # workshop of [artist]
                formed = a.get("formed_by") or {}
                for infl in as_list(formed.get("influenced_by")):
                    if infl.get("id") and infl.get("type") == "Person":
                        bag.append((infl["id"], "workshop_of"))

    pb = cho_ld.get("produced_by") or {}
    candidates = []
    add_from_cob(pb.get("carried_out_by"), candidates)
    for aa in as_list(pb.get("assigned_by")):
        add_from_assignment(aa, candidates)
    for part in as_list(pb.get("part")):
        add_from_cob(part.get("carried_out_by"), candidates)
        for aa in as_list(part.get("assigned_by")):
            add_from_assignment(aa, candidates)

    # dedupe but keep first qualifier
    seen, order = {}, []
    for pid, qual in candidates:
        if pid not in seen:
            seen[pid] = qual
            order.append(pid)

    people = []
    for pid in order:
        info = read_rijksmuseum_person(pid, depth=1)
        if info:
            if seen[pid]:
                info = dict(info); info["qualifier"] = seen[pid]
            people.append(info)

    return people[0] if len(people) == 1 else people

# Descriptions

# Needed to filter out porvenance texts
def _looks_like_provenance_text(txt):
    t = txt.strip()
    if t.startswith("…") or t.startswith("..."): return True
    if "{" in t and "}" in t: return True
    if t.count(";") >= 2: return True
    return False

def pick_descriptions(cho_ld):
    out = {}

    # Placed in subject_of
    for so in (cho_ld.get("subject_of") or []):
        for lo, langs in walk_lo_with_lang(so, so.get("language")):
            if classified_ids(lo) & DESC_AAT:
                raw = (lo.get("content") or "").strip()
                if not raw:
                    continue
                txt = strip_html(raw)
                if not txt:
                    continue
                for tag in lang_tags(langs):
                    if tag not in out:
                        out[tag] = txt

    # If still missing for any language
    missing = {"English", "Dutch"} - set(out.keys())
    if missing:
        EXCLUDE = set().union(DIM_AAT, MED_AAT, DESC_AAT, CRED_AAT, PROV_AAT, INSCRIPTION_AAT)
        candidates = []
        for lo in (cho_ld.get("referred_to_by") or []):
            if not isinstance(lo, dict):
                continue
            kinds = classified_ids(lo)
            if kinds & EXCLUDE:
                continue
            raw = (lo.get("content") or "").strip()
            if len(raw) < 40:
                continue
            txt = strip_html(raw)
            if not txt or _looks_like_provenance_text(txt):
                continue

            rank = 0 if (kinds & LABEL_AAT) else 1
            candidates.append((rank, lo, txt))

        candidates.sort(key=lambda x: (x[0], -len(x[2])))
        for _, lo, txt in candidates:
            for tag in lang_tags(lo.get("language")):
                if tag in missing and tag not in out:
                    out[tag] = txt
                    missing.discard(tag)
            if not missing:
                break

    return out or None

# Date
YEAR_SPAN_RE = re.compile(r'(?i)(?:c(?:a|irca)\.?\s*)?(\d{3,4})(?:\s*[-–]\s*(\d{3,4}))?')

def _format_date_label(beg_iso, end_iso):
    y1 = beg_iso[:4] if isinstance(beg_iso, str) and len(beg_iso) >= 4 else None
    y2 = end_iso[:4] if isinstance(end_iso, str) and len(end_iso) >= 4 else None
    if y1 and y2 and y1 != y2: return f"{y1}–{y2}"
    return y1 or y2

def pick_date(cho_ld):
    pb = cho_ld.get("produced_by") or {}
    spans = []
    spans += as_list(pb.get("timespan"))
    for part in as_list(pb.get("part")):
        spans += as_list(part.get("timespan"))
    for lo in as_list(pb.get("referred_to_by")):
        spans += as_list(lo.get("timespan"))

    iso = [(ts.get("begin_of_the_begin"), ts.get("end_of_the_end")) 
           for ts in spans if isinstance(ts, dict) and (ts.get("begin_of_the_begin") or ts.get("end_of_the_end"))]
    if iso:
        b = min([b for b, _ in iso if b] or [None])
        e = max([e for _, e in iso if e] or [None])
        lab = _format_date_label(b, e)
        if lab: return lab

    years = []
    for ts in spans:
        for ident in as_list(ts.get("identified_by") if isinstance(ts, dict) else None):
            m = YEAR_SPAN_RE.search((ident.get("content") or ""))
            if m: years.append((m.group(1), m.group(2)))
    if years:
        y1s = [int(a) for a,_ in years if a]
        y2s = [int(b) for _,b in years if b]
        beg = f"{min(y1s):04d}-01-01T00:00:00Z" if y1s else None
        end = f"{max((y2s or y1s)) :04d}-12-31T23:59:59Z" if (y2s or y1s) else None
        return _format_date_label(beg, end)
    return None

# Type
def pick_type(cho_ld):
    for t in (cho_ld.get("classified_as") or []):
        lab = t.get("_label") or t.get("label")
        if lab: return lab
        tid = t.get("id")
        if tid:
            lab = read_rijksmuseum_concept(tid)
            if lab: return lab
    return None

# Medium
def pick_mediums(cho_ld):
    meds = []
    for m in (cho_ld.get("made_of") or []):
        lab = None
        for idb in (m.get("identified_by") or []):
            if idb.get("type") == "Name" and idb.get("content"):
                lab = idb["content"]; break
        if not lab and m.get("id"):
            lab = read_rijksmuseum_concept(m["id"])
        if lab:
            meds.append(lab)

    return merge_multilang_dicts(meds) or None

# TO DO: Check Extent
def pick_extent(cho_ld):
    
    def _lang_rank(tags):
        ids = [(t.get("id") or "") for t in (tags or [])]
        if AAT_LANG_EN in ids: return 0
        if AAT_LANG_NL in ids: return 1
        return 2

    def _idnum(lo, default=999):
        for idb in as_list(lo.get("identified_by")):
            if idb.get("type") == "Identifier":
                try:
                    return int(str(idb.get("content")).strip())
                except Exception:
                    pass
        return default

    def _unit_label(u):
        return UNIT_LABELS.get(((u or {}).get("id") or ""), "")

    los = [lo for lo in as_list(cho_ld.get("referred_to_by")) if classified_ids(lo) & DIM_AAT]

    def is_outer(txt):  # skip outer/frame
        t = txt.lower()
        return ("buitenmaat" in t) or ("outer size" in t) or (" frame" in t)

    def is_base(txt):   # base/footprint lines
        t = txt.lower()
        return ("voet" in t) or (" base" in t)

    def has_hw(txt):
        t = txt.lower()
        return (("height" in t) or ("hoogte" in t)) and (("width" in t) or ("breedte" in t))

    def has_height_only(txt):
        t = txt.lower()
        return (("height" in t) or ("hoogte" in t)) and not (("width" in t) or ("breedte" in t) or ("length" in t) or ("lengte" in t))

    # Collect candidates by role
    both_hw, height_only, base_len_wid = [], [], []
    for lo in los:
        txt = (lo.get("content") or "").strip()
        if not txt or is_outer(txt):
            continue
        key = (_lang_rank(lo.get("language")), _idnum(lo), -len(txt))
        if is_base(txt):
            base_len_wid.append((key, txt))
        elif has_hw(txt):
            both_hw.append((key, txt))
        elif has_height_only(txt):
            height_only.append((key, txt))

    if both_hw:
        both_hw.sort(key=lambda x: x[0])
        return both_hw[0][1]

    if height_only and base_len_wid:
        height_only.sort(key=lambda x: x[0])
        base_len_wid.sort(key=lambda x: x[0])
        htxt = height_only[0][1].lower()
        btxt = base_len_wid[0][1].lower()

        # extract numbers+units crudely
        num_re = r"(\d+(?:[.,]\d+)?)\s*(cm|mm|m)"
        h_m = re.search(r"(hoogte|height)\s*" + num_re, htxt)
        l_m = re.search(r"(lengte|length)\s*" + num_re, btxt)
        w_m = re.search(r"(breedte|width)\s*" + num_re, btxt)

        def norm(v):  # replace comma with dot
            return v.replace(",", ".") if isinstance(v, str) else v

        parts = []
        if h_m:
            parts.append(f"height {norm(h_m.group(2))} {h_m.group(3)}")
        if l_m:
            parts.append(f"length {norm(l_m.group(2))} {l_m.group(3)}")
        if w_m:
            parts.append(f"width {norm(w_m.group(2))} {w_m.group(3)}")
        if parts:
            return " x ".join(parts)

    height = length = width = thickness = None
    uh = ul = uw = ut = ""

    for d in as_list(cho_ld.get("dimension")):
        names = [(idb.get("content") or "").lower()
                 for idb in as_list(d.get("identified_by"))
                 if idb.get("type") == "Name"]
        nm = " ".join(names)

        if ("buitenmaat" in nm) or ("outer size" in nm):
            continue

        val = (d.get("value") or "")
        unit = _unit_label(d.get("unit") or {})

        if "voet" in nm or "base" in nm:
            if ("lengte" in nm or "length" in nm):
                length, ul = val, unit
            if ("breedte" in nm or "width" in nm):
                width, uw = val, unit
            continue

        if ("hoogte" in nm) or ("height" in nm):
            height, uh = val, unit
        elif ("breedte" in nm) or ("width" in nm):
            width, uw = val, unit
        elif ("diepte" in nm) or ("depth" in nm):
            thickness, ut = val, unit
        elif ("dikte" in nm) or ("thickness" in nm):
            thickness, ut = val, unit
        elif not names:
            if val and unit and not height:
                height, uh = val, unit

    parts = []
    if height and uh: parts.append(f"height {height} {uh}")
    if length and ul: parts.append(f"length {length} {ul}")
    if width  and uw: parts.append(f"width {width} {uw}")

    if not length and thickness and ut:
        parts.append(f"depth {thickness} {ut}")

    return " x ".join(parts) if parts else None

# Extent with frame
def pick_extent_with_frame(cho_ld):
    for lo in as_list(cho_ld.get("referred_to_by")):
        txt = (lo.get("content") or "").strip()
        if not txt: continue
        low = txt.lower()
        if any(k in low for k in ("frame","depth","diepte")):
            return txt
    return None

# Subjects
def ensure_multilang_labels(x):
    if isinstance(x, str): return {"und": [x]}
    if isinstance(x, dict):
        return {k: [v for v in (vals or []) if isinstance(v, str) and v.strip()]
                for k, vals in x.items() if (vals or [])}
    return {}

def pick_subjects(cho_ld):
    pieces = []
    for s in as_list(cho_ld.get("shows")):
        for it in as_list(s.get("represents_instance_of_type")) + as_list(s.get("about")):
            if isinstance(it, dict) and it.get("id"):
                labs = read_rijksmuseum_concept(it["id"]) or {}
                pieces.append(labs)
            else:
                lab = (isinstance(it, dict) and (it.get("_label") or it.get("label"))) or (it if isinstance(it, str) else None)
                if lab: pieces.append({"und": [lab]})
    return merge_multilang_dicts([ensure_multilang_labels(p) for p in pieces]) or None

# Place
def _resolve_place_node(pl):
    if isinstance(pl, str):
        return read_rijksmuseum_place(pl)
    if isinstance(pl, dict):
        lab = pl.get("_label") or pl.get("label")
        if lab:
            return lab
        if pl.get("id"):
            return read_rijksmuseum_place(pl["id"])
    return None

def pick_place(cho_ld):
    pb = cho_ld.get("produced_by") or {}
    for part in pb.get("part") or []:
        for pl in part.get("took_place_at") or []:
            lab = _resolve_place_node(pl)
            if lab:
                return lab
    for s in cho_ld.get("shows") or []:
        for pl in s.get("represents") or []:
            lab = _resolve_place_node(pl)
            if lab:
                return lab
    return None

# Current location
def _looks_like_room_or_code(txt):
    t = txt.lower()
    return any(ch.isdigit() for ch in t) or "-" in t or t.startswith(("hg", "zaal", "room"))

def pick_current_location(cho_ld):
    loc = cho_ld.get("current_location") or {}
    if isinstance(loc, str):
        lab = _resolve_place_node(loc)
        if lab: return lab
    elif isinstance(loc, dict) and loc.get("id"):
        lab = _resolve_place_node(loc["id"])
        if lab: return lab

    name_candidates = []
    for idb in as_list(loc.get("identified_by")):
        for part in as_list(idb.get("part")):
            if part.get("type") == "Name" and part.get("content"):
                name_candidates.append(part)
        if idb.get("type") == "Name" and idb.get("content"):
            name_candidates.append(idb)

    for it in name_candidates:
        txt = (it.get("content") or "").strip()
        if txt and not _looks_like_room_or_code(txt): 
            return txt
    for it in name_candidates:
        txt = (it.get("content") or "").strip()
        if txt: return txt
    return None

# Identifiers
def pick_identifier(cho_ld):
    for idb in as_list(cho_ld.get("identified_by")):
        if idb.get("type") == "Identifier" and idb.get("content"):
            return idb["content"]
    return None

# URI
def pick_cho_uri(cho_ld):
    return cho_ld.get("id")

# Rights
def pick_rights(cho_ld):
    for s in as_list(cho_ld.get("shows")):
        for r in as_list(s.get("subject_to")):
            for c in as_list(r.get("classified_as")):
                return c.get("_label") or c.get("label") or c.get("id")
    for so in as_list(cho_ld.get("subject_of")):
        for r in as_list(so.get("subject_to")):
            for c in as_list(r.get("classified_as")):
                return c.get("_label") or c.get("label") or c.get("id")
    return None

# Inscriptions
def _group_lang_text(items):
    out = {"English": [], "Dutch": []}
    for it in items or []:
        if not isinstance(it, dict): continue
        txt = (it.get("content") or "").strip()
        if not txt: continue
        lc = lang_code(it)
        if lc in out: out[lc].append(txt)
    return {k:v for k,v in out.items() if v}

def pick_inscriptions(cho_ld):
    los = []
    for lo in as_list(cho_ld.get("referred_to_by")):
        if classified_ids(lo) & INSCRIPTION_AAT and (lo.get("content") or "").strip():
            los.append(lo)
    return _group_lang_text(los) or None

# Set memberships
def pick_set_memberships(cho_ld):
    ids = [s.get("id") for s in as_list(cho_ld.get("member_of")) if isinstance(s, dict) and s.get("id")]
    out = {"ids": ids}
    return out or None

### Notes:
```set_membership``` can be very important for those sets that "break the query", like Entire Public Domain Set. It could be possible to sample from another set and filter out those that are not Public. As well, to get more information about the artwork. 

In [326]:
IDS = ["https://id.rijksmuseum.nl/200105887", 
       "https://id.rijksmuseum.nl/200100988", 
       "https://id.rijksmuseum.nl/200109279",
       "https://id.rijksmuseum.nl/200108412",
       "https://id.rijksmuseum.nl/200108160",
       "https://id.rijksmuseum.nl/200107815",
       "https://id.rijksmuseum.nl/200109287",
       "https://id.rijksmuseum.nl/20023687",
       "https://id.rijksmuseum.nl/20027707",
       "https://id.rijksmuseum.nl/200106308"]

def build_row(id):
    cho_ld = fetch_ld(id)
    return {
        "title": pick_titles(cho_ld),
        "artist_info": pick_artist_info(cho_ld),
        "description": pick_descriptions(cho_ld),
        "date": pick_date(cho_ld),
        "arttype": pick_type(cho_ld),
        "medium": pick_mediums(cho_ld),
        "extent": pick_extent(cho_ld),
        "extent_with_frame": pick_extent_with_frame(cho_ld),
        "subjects": pick_subjects(cho_ld),
        "place": pick_place(cho_ld),
        "current_location": pick_current_location(cho_ld),
        "inscriptions": pick_inscriptions(cho_ld),
        "set_membership": pick_set_memberships(cho_ld),
        "identifier": pick_identifier(cho_ld),
        "cho_uri": pick_cho_uri(cho_ld),
        "rights": pick_rights(cho_ld),

        "image": None,
    }

res = build_row(IDS[7])
print(res)

{'title': {'English': ['Guanyin Crossing the Sea', 'Figure of a standing Guanyin'], 'Dutch': ['Beeld van een staande Guanyin', 'Guanyin doorkruist de zee']}, 'artist_info': {'id': 'https://id.rijksmuseum.nl/210823', 'names_by_lang': {'Dutch': ['He Chaozong']}, 'birth': {'date': None, 'place_by_lang': None}, 'death': {'date': None, 'place_by_lang': None}, 'activities_by_lang': {}, 'equivalent': []}, 'description': {'Dutch': 'Deze grote sculptuur van blanc de Chine (wit porselein) toont Guanyin staand op de golven, waarmee haar bovennatuurlijke kracht wordt uitgedrukt. Het beeld wordt toegeschreven aan He Chaozong, de beste pottenbakker van het gebied Dehua en beroemd om zijn elegante en goed gemodelleerde figuren.', 'English': 'This large blanc de Chine (whiteware) figure shows Guanyin standing on the waves, underscoring her supernatural powers. The statue is attributed to He Chaozong, the best potter in the county of Dehua, famed for his elegant and beautifully modelled figures.'}, 'da

### Extract image + missing information from RDF

In [339]:
def _is_empty(x):
    if x is None: return True
    if isinstance(x, (list, dict, set)) and not x: return True
    if isinstance(x, str) and not x.strip(): return True
    return False


def _lang_from_xml(elem):
    lang = (elem.attrib.get('{http://www.w3.org/XML/1998/namespace}lang') or '').lower()
    if lang.startswith('en'): return 'English'
    if lang.startswith('nl'): return 'Dutch'
    return None

def _collect_literals(rdf, xpath):
    out = []
    for el in rdf.findall(xpath, NS):
        txt = (el.text or '').strip()
        if txt:
            out.append((txt, _lang_from_xml(el)))
    return out

def _first_resource_attr(nodes):
    for n in nodes:
        res = n.attrib.get('{%s}resource' % NS['rdf'])
        if res: return res
    return None

def _about_index(rdf):
    idx = {}
    rdfns = '{%s}about' % NS['rdf']
    for el in rdf.findall(".//*[@rdf:about]", NS):
        about = el.attrib.get(rdfns)
        if about:
            idx[about] = el
    return idx

def _literal_with_lang(el):
    if el is None:
        return None
    txt = (el.text or "").strip()
    if not txt:
        return None
    lang = (
        el.attrib.get('{http://www.w3.org/XML/1998/namespace}lang')
        or el.attrib.get('lang')
    )
    return (txt, lang)

def _label_children(el):
    out = []
    for xp in (
        ".//skos:prefLabel",
        ".//rdfs:label",
        ".//dc:title",
        ".//skos:altLabel",
        ".//prefLabel",  # some dumps omit skos:
    ):
        for lab in el.findall(xp, NS):
            pair = _literal_with_lang(lab)
            if pair:
                out.append(pair)
        if out:
            break
    return out

def _collect_resolved(rdf, xpath):
    idx = _about_index(rdf)
    out = []
    res_attr = '{%s}resource' % NS['rdf']

    for node in rdf.findall(xpath, NS):
        # literal directly
        lit = _literal_with_lang(node)
        if lit:
            out.append(lit)
            continue

        # or resource reference
        uri = node.attrib.get(res_attr)
        if uri and uri in idx:
            out.extend(_label_children(idx[uri]))
    return out

def fill_row_from_rdf(row, rdf):
    if _is_empty(row.get("image")):
        aggr = rdf.find(".//ore:Aggregation", NS)
        image = None
        if aggr is not None:
            wr = aggr.find(".//edm:isShownBy/edm:WebResource", NS)
            if wr is not None:
                image = wr.attrib.get(f"{{{NS['rdf']}}}about")
        if image:
            row["image"] = image or row.get("image")
            return
        
        nodes = rdf.findall(".//edm:isShownBy", NS)
        img = _first_resource_attr(nodes)
        if not img:
            for wr in rdf.findall(".//edm:WebResource", NS):
                about = wr.attrib.get('{%s}about' % NS['rdf'])
                fmts  = [ (f.text or '').lower() for f in wr.findall(".//dc:format", NS) ]
                if about and any(("image" in f or f.startswith('image/')) for f in fmts):
                    img = about; break
        row["image"] = img or row.get("image")

    # Title
    if _is_empty(row.get("title")):
        titles = _collect_literals(rdf, ".//dc:title")
        ml = {}
        for txt, lg in titles:
            if not lg: continue
            ml.setdefault(lg, []).append(txt)
        row["title"] = {k: list(dict.fromkeys(v)) for k, v in ml.items()} or row.get("title")

    # Artist Info
    if _is_empty(row.get("artist_info")):
        creators = _collect_literals(rdf, ".//dc:creator")
        # Minimal person schema fallback: [{'label':'...'}]
        if creators:
            row["artist_info"] = [{"label": txt} for (txt, _lg) in creators]

    # Date
    if _is_empty(row.get("date")):
        # Prefer dcterms:created/date, else dc:date
        for xp in (".//dcterms:created", ".//dcterms:date", ".//dc:date"):
            dates = _collect_literals(rdf, xp)
            if dates:
                row["date"] = dates[0][0]
                break

    # Type
    if _is_empty(row.get("arttype")):
        types = _collect_resolved(rdf, ".//edm:ProvidedCHO/dc:type | .//edm:ProvidedCHO/edm:type")
        if types:
            # prefer English label if available
            types_sorted = sorted(types, key=lambda p: (0 if (p[1] and p[1].startswith("en")) else 1, -len(p[0])))
            row["arttype"] = types_sorted[0][0]

    # Medium
    if _is_empty(row.get("medium")):
        meds = _collect_resolved(rdf, ".//edm:ProvidedCHO/dc:medium")
        if meds:
            ml = {"English": [], "Dutch": []}
            for txt, lg in meds:
                if lg and lg.startswith("en"):
                    ml["English"].append(txt)
                elif lg and lg.startswith("nl"):
                    ml["Dutch"].append(txt)
            # dedupe & prune
            ml = {k: list(dict.fromkeys(v)) for k,v in ml.items() if v}
            row["medium"] = ml or row.get("medium")
    # Extent
    if _is_empty(row.get("extent")):
        exts = _collect_literals(rdf, ".//dcterms:extent")
        if exts:
            row["extent"] = exts[0][0]

    # Subjects
    if _is_empty(row.get("subjects")):
        subs = []
        # from CHO
        subs.extend(_collect_resolved(rdf, ".//edm:ProvidedCHO/dc:subject"))
        # some providers also repeat subjects on the Aggregation
        subs.extend(_collect_resolved(rdf, ".//edm:Aggregation/dc:subject"))

        if subs:
            ml = {}
            for txt, lg in subs:
                key = ("English" if (lg and lg.startswith("en")) 
                    else "Dutch" if (lg and lg.startswith("nl")) 
                    else "und")
                ml.setdefault(key, []).append(txt)
            # dedupe
            row["subjects"] = {k: list(dict.fromkeys(v)) for k, v in ml.items()}
    # PLace
    if _is_empty(row.get("place")):
        # dcterms:spatial often used; fallback dc:coverage
        places = _collect_literals(rdf, ".//dcterms:spatial")
        if not places:
            places = _collect_literals(rdf, ".//dc:coverage")
        if places:
            row["place"] = places[0][0]

    # ID
    if _is_empty(row.get("identifier")):
        ids = _collect_literals(rdf, ".//dc:identifier")
        if ids:
            row["identifier"] = ids[0][0]

    # Rights
    if _is_empty(row.get("rights")):
        # edm:rights is often a URI @rdf:resource; dc:rights may be literal
        rn = rdf.findall(".//edm:rights", NS)
        rights_uri = _first_resource_attr(rn)
        if rights_uri:
            row["rights"] = rights_uri
        else:
            rights_lit = _collect_literals(rdf, ".//dc:rights")
            if rights_lit:
                row["rights"] = rights_lit[0][0]

    return row

def _rdf_resource(node):
    if node is None:
        return None
    return node.attrib.get('{%s}resource' % NS['rdf'])

def _rdf_about(node):
    if node is None:
        return None
    return node.attrib.get('{%s}about' % NS['rdf'])

def extract_record(rdf):
    cho_uri = None

    agg = rdf.find(".//ore:Aggregation", NS)
    if agg is not None:
        cho_uri = _rdf_resource(agg.find("edm:aggregatedCHO", NS))

    # Fallback: edm:ProvidedCHO/@rdf:about
    if not cho_uri:
        prov_cho = rdf.find(".//edm:ProvidedCHO", NS)
        cho_uri = _rdf_about(prov_cho)

    # 3) Build the row from JSON-LD (or an empty shell if fetch failed)
    row = build_row(cho_uri)
    row["cho_uri"] = cho_uri  # ensure URI is set, even if JSON-LD was missing

    # 4) Backfill any empty cells from the EDM/RDF
    fill_row_from_rdf(row, rdf)

    return row, cho_uri

### Extract one record completely

In [342]:
# set might break the query.
root = fetch({"verb":"ListRecords", "metadataPrefix":"edm", "set":"260213"})

rec = root.find(".//oai:record", NS)
md = rec.find(".//oai:metadata", NS)
rdf = md.find(".//rdf:RDF", NS)

row, cho_uri = extract_record(rdf)


### Extract the first 100 records from particular sets and store them in a CSV

In [ ]:
import pandas as pd

set_ids = ["261233", "261231", "26128", "260235", "26147", '26126', "2618", "260239"]

def harvest_set(set_id, n):
    rows = []
    token = None
    
    while len(rows) < n:
        args = {"verb":"ListRecords", "metadataPrefix":"edm", "set":set_id}
        if token:
            args = {"verb":"ListRecords", "resumptionToken": token}
        root = fetch(args)

        for rec in root.findall(".//oai:record", NS):
            if len(rows) >= n:
                break
            md = rec.find(".//oai:metadata", NS)
            if md is None:
                continue
            rdf = md.find(".//rdf:RDF", NS)
            if rdf is None:
                continue

            try:
                row, cho_uri = extract_record(rdf, NS)
                row["cho_uri"] = cho_uri  # keep track
                rows.append(row)
            except Exception as e:
                print("Error extracting record:", e)

        # handle resumptionToken
        rt = root.find(".//oai:resumptionToken", NS)
        token = rt.text.strip() if rt is not None and rt.text else None
        if not token:
            break  # no more records

    # build DataFrame
    df = pd.DataFrame(rows)
    return df

for set_id in set_ids:
    out_csv = f"{CACHE_DIR}/rijks_{set_id}_100.csv"
    harvest_set(set_id, 100).to_csv(out_csv, index=False)